# Spark Performance — Execution Plan Diagnostics

Demonstrate three anti-patterns on silver tables. For each: run **bad** vs **fixed** query, compare `explain()` output and timing, then capture **Query Profile** screenshots in the Databricks UI (run cell → Spark UI → SQL/DataFrame → View Query Profile).

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.spark_performance.execution_plans as execution_plans_module
importlib.reload(execution_plans_module)

from src.spark_performance.execution_plans import (
    SilverTableRefs,
    capture_explain,
    benchmark_df,
    predicate_after_join_bad,
    predicate_before_join_good,
    join_without_broadcast_bad,
    join_with_broadcast_good,
    repartition_before_filter_bad,
    filter_before_join_good,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverTableRefs()
print("Loaded from:", execution_plans_module.__file__)

## 1. Predicate after expensive ops (filter after join + groupBy)

**Bad:** join all order items → orders → customers, aggregate by state, then filter `SP`.

**Fix:** filter customers to `SP` first, then join and aggregate.

In [ ]:
bad_1 = predicate_after_join_bad(spark, tables)
good_1 = predicate_before_join_good(spark, tables)

print("=== BAD explain ===")
print(capture_explain(bad_1))
print("=== GOOD explain ===")
print(capture_explain(good_1))

timing_1 = {"bad": benchmark_df(bad_1), "good": benchmark_df(good_1)}
timing_1["speedup_x"] = round(timing_1["bad"]["elapsed_ms"] / timing_1["good"]["elapsed_ms"], 2)
print("Timing:", timing_1)

## 2. Wrong join strategy (sort-merge instead of broadcast)

**Bad:** `hint("merge")` forces sort-merge join order_items (~112k) with sellers (~3k).

**Fix:** `broadcast(sellers)` so the small table ships to executors.

*Databricks Free blocks `spark.conf.set("spark.sql.autoBroadcastJoinThreshold")` — join hints are used instead.*

In [ ]:
bad_2 = join_without_broadcast_bad(spark, tables)
good_2 = join_with_broadcast_good(spark, tables)

print("=== BAD explain (look for SortMergeJoin / Exchange) ===")
print(capture_explain(bad_2))
print("=== GOOD explain (look for BroadcastHashJoin) ===")
print(capture_explain(good_2))

timing_2 = {"bad": benchmark_df(bad_2), "good": benchmark_df(good_2)}
timing_2["speedup_x"] = round(timing_2["bad"]["elapsed_ms"] / timing_2["good"]["elapsed_ms"], 2)
print("Timing:", timing_2)

## 3. Unnecessary shuffle (repartition before filter)

**Bad:** `repartition(200, order_id)` then filter — forces a shuffle before reducing rows.

**Fix:** filter on price first, then join without forced repartition.

In [ ]:
bad_3 = repartition_before_filter_bad(spark, tables)
good_3 = filter_before_join_good(spark, tables)

print("=== BAD explain (look for Exchange / Repartition) ===")
print(capture_explain(bad_3))
print("=== GOOD explain ===")
print(capture_explain(good_3))

timing_3 = {"bad": benchmark_df(bad_3), "good": benchmark_df(good_3)}
timing_3["speedup_x"] = round(timing_3["bad"]["elapsed_ms"] / timing_3["good"]["elapsed_ms"], 2)
print("Timing:", timing_3)

In [ ]:
import json

report = {
    "task": "2.1_execution_plan_diagnostics",
    "patterns": [
        {
            "name": "predicate_after_expensive_ops",
            "timing": timing_1,
            "bad_plan_snippet": capture_explain(bad_1).splitlines()[:12],
            "good_plan_snippet": capture_explain(good_1).splitlines()[:12],
        },
        {
            "name": "wrong_join_strategy",
            "timing": timing_2,
            "bad_plan_snippet": capture_explain(bad_2).splitlines()[:12],
            "good_plan_snippet": capture_explain(good_2).splitlines()[:12],
        },
        {
            "name": "unnecessary_shuffle_repartition",
            "timing": timing_3,
            "bad_plan_snippet": capture_explain(bad_3).splitlines()[:12],
            "good_plan_snippet": capture_explain(good_3).splitlines()[:12],
        },
    ],
    "note": "Attach Query Profile screenshots separately for grading deliverable.",
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/m02_task21_execution_plans.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)